# SANA A-PSL: Zero-Assumption Dataset Probing & Landmark Inspection
**Goal:** Inspect and verify candidate Pakistani Sign Language (PSL) datasets (`Bakhtyar12` on HuggingFace and `wordlevel-pakistan-sign-language-dataset` on Kaggle) without making assumptions.
- Check exact file hierarchy, `.npy` array shapes, NaN values, and landmark indexing.
- Verify 208-dimension alignment to our pre-trained SANA Conv1D SpatialTemporal Foundation Model.

In [ ]:
# ── Cell 1: Environment Setup & Dependencies ─────────────────────────────────
!pip install -q datasets huggingface_hub mediapipe matplotlib pandas numpy
import os, glob, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("Environment ready for PSL dataset inspection.")

In [ ]:
# ── Cell 2: Probe Kaggle Dataset (ahmedabid/mohibkhan 70 PSL Words) ──────────
print("=" * 80)
print("PROBING KAGGLE PSL DATASET: /kaggle/input")
print("=" * 80)

kaggle_psl_dirs = []
for root, dirs, files in os.walk("/kaggle/input"):
    npy_files = [f for f in files if f.endswith('.npy') or f.endswith('.csv')]
    if len(npy_files) > 0:
        kaggle_psl_dirs.append((root, len(npy_files), npy_files[:3]))

if kaggle_psl_dirs:
    for root_dir, count, samples in kaggle_psl_dirs[:10]:
        print(f"Directory: {root_dir} -> Found {count} data files.")
        print(f"  Sample files: {samples}")
else:
    print("Note: To inspect Kaggle PSL data, click '+ Add Input' -> search 'wordlevel-pakistan-sign-language-dataset'.")

In [ ]:
# ── Cell 3: Probe HuggingFace Dataset (Bakhtyar12/Pakistani-Sign-Language) ───
print("=" * 80)
print("PROBING HUGGING FACE PSL DATASET: Bakhtyar12/Pakistani-Sign-Language")
print("=" * 80)

try:
    from datasets import load_dataset
    print("Loading dataset metadata from Hugging Face...")
    ds = load_dataset("Bakhtyar12/Pakistani-Sign-Language", split="train", streaming=True)
    first_sample = next(iter(ds))
    print("Successfully fetched live streaming sample from Hugging Face!")
    print(f"Keys available in dataset: {list(first_sample.keys())}")
    for k, v in first_sample.items():
        if isinstance(v, (list, np.ndarray)):
            arr = np.array(v)
            print(f"  Field '{k}': shape {arr.shape}, dtype {arr.dtype}, min {arr.min():.3f}, max {arr.max():.3f}")
        else:
            print(f"  Field '{k}': {v}")
except Exception as e:
    print(f"HuggingFace dataset probe note: {e}")

In [ ]:
# ── Cell 4: Zero-Mismatch 208-Dimension Adapter Validation ──────────────────
def adapt_psl_to_208dim(raw_landmarks, target_frames=100):
    """
    Adapts any raw MediaPipe landmark sequence (150-dim or 225-dim)
    into our exact 208-dimensional SANA foundation tensor format.
    
    Layout (208 floats per frame):
    - [0:66]   : 33 Pose landmarks (X, Y)
    - [66:108] : 21 Left Hand landmarks (X, Y)
    - [108:150]: 21 Right Hand landmarks (X, Y)
    - [150:208]: 29 Expression Face landmarks (Neutral 0s)
    """
    raw = np.array(raw_landmarks, dtype=np.float32)
    T = raw.shape[0]
    
    # If 225-dim 3D (75 points x 3), extract (x, y)
    if raw.shape[-1] == 225:
        raw_reshaped = raw.reshape(T, 75, 3)
        pose_2d = raw_reshaped[:, 0:33, :2].reshape(T, 66)
        lh_2d   = raw_reshaped[:, 33:54, :2].reshape(T, 42)
        rh_2d   = raw_reshaped[:, 54:75, :2].reshape(T, 42)
        manual_150 = np.concatenate([pose_2d, lh_2d, rh_2d], axis=1)
    elif raw.shape[-1] == 150:
        manual_150 = raw
    else:
        manual_150 = raw[:, :150]
        
    # Attach 58 neutral face coordinates
    neutral_face_58 = np.zeros((T, 58), dtype=np.float32)
    adapted_208 = np.concatenate([manual_150, neutral_face_58], axis=1)
    
    # Pad / Truncate to target_frames
    if T >= target_frames:
        final_seq = adapted_208[:target_frames]
        valid_len = target_frames
    else:
        final_seq = np.zeros((target_frames, 208), dtype=np.float32)
        final_seq[:T] = adapted_208
        valid_len = T
        
    return final_seq, valid_len

# Test with dummy 30-frame sequence (shape 30, 225)
dummy_raw = np.random.randn(30, 225).astype(np.float32)
adapted_tensor, valid_len = adapt_psl_to_208dim(dummy_raw, target_frames=100)
print("=" * 80)
print("DIMENSION ALIGNMENT TEST RESULTS:")
print(f"  Input Raw Shape:    {dummy_raw.shape}")
print(f"  Adapted SANA Shape: {adapted_tensor.shape} (Matches Conv1D Input: 208!)")
print(f"  Valid Frame Count:  {valid_len} frames")
print("=" * 80)